<a href="https://colab.research.google.com/github/Sciform/sciform-hwz-ai-in-controlling/blob/main/lecture_3/3_4_ai_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents for Controlling: a Month-End Closing Team with Google ADK and Ollama

An **AI agent** is a language model (LLM) that does not only answer with text, but can **use tools** – here Python functions – to
retrieve data, calculate figures and then decide on the next step.

In this notebook we build a small agent team that supports the **month-end closing** of an online shop:

```
"Prepare the month-end report for 2011-11"
        │
        ▼
┌────────────────┐    ┌──────────────────┐    ┌───────────────────┐
│ 1. KPI analyst │ ─▶ │ 2. Auditor       │ ─▶ │ 3. Report writer  │
│ key figures,   │    │ cancellations,   │    │ management        │
│ variance       │    │ manual postings, │    │ commentary        │
│ analysis       │    │ unusual orders   │    │                   │
└────────────────┘    └──────────────────┘    └───────────────────┘
```

**Technology**

* [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/) – open-source Python framework for agents and multi-agent systems.
* [Ollama](https://ollama.com) – runs open-source LLMs **locally**: no API key, no costs, and no company data leaves the computer –
  an important argument for controlling and finance data. ADK talks to Ollama via [LiteLLM](https://docs.litellm.ai).
* Model: `qwen2.5:7b` (Alibaba, Apache 2.0) – a small open model that is good at tool calling.

**The most important design principle:** the LLM never calculates figures itself. All figures come from ordinary, tested Python functions
(the tools). The LLM decides *which* tool to call and *explains* the results. Thus every number in the report can be traced back.

> **Colab:** please use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). On a CPU the model works too, but is very slow.

**Agenda**

1. Set up Ollama and the model
2. Load the *Online Retail* data
3. Write and test the tools – plain Python, no AI
4. A single agent: the controlling assistant
5. The agent team: month-end report with a `SequentialAgent`
6. Exercises

## 1. Setup

### Install Ollama and start the server

The following cells install Ollama on the Colab machine (Linux). On your own computer, install Ollama from
[ollama.com](https://ollama.com) instead and skip these cells.

In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

# start the Ollama server in the background; a context of 8k tokens is needed for tool results and the agent history
ollama_server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_CONTEXT_LENGTH": "8192"},
                                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

In [ ]:
MODEL = "qwen2.5:7b"     # alternatives: "llama3.1:8b", "qwen3:8b"; for a CPU without GPU: "qwen2.5:3b"
!ollama pull {MODEL}

In [ ]:
# mcp: needed by current LiteLLM versions, but not installed automatically
%pip install -q google-adk litellm mcp

In [ ]:
import json
import warnings
import logging

import pandas as pd
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"
llm = LiteLlm(model=f"ollama_chat/{MODEL}")     # "ollama_chat" supports tool calls

## 2. Load the data

The *Online Retail* data set contains all invoice lines of a UK online gift shop from Dec 2010 to Dec 2011 (see lecture 2).
For the month-end closing we need **all** lines, including those we removed for forecasting:

* **cancellations** – `InvoiceNo` starts with `C`, negative quantity,
* **non-product postings** – postage (`POST`, `DOT`, `C2`), fees (`AMAZONFEE`, `BANK CHARGES`), manual postings (`M`), discounts (`D`),
  bad debt adjustments (`B`), samples (`S`), ...

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sciform/sciform-hwz-ai-in-controlling/main/data/online_retail.csv

In [ ]:
data = pd.read_csv("online_retail.csv", dtype={"InvoiceNo": str, "StockCode": str}, encoding="ISO-8859-1")
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], format="%m/%d/%Y %H:%M")
data["Value"] = data["Quantity"] * data["UnitPrice"]
data["Month"] = data["InvoiceDate"].dt.to_period("M").astype(str)
data["IsCancellation"] = data["InvoiceNo"].str.startswith("C")
data["IsProduct"] = data["StockCode"].str.match(r"^\d{5}")      # real products have a 5-digit stock code

POSTING_TYPES = {"POST": "shipping", "DOT": "shipping", "C2": "shipping",
                 "AMAZONFEE": "fees", "BANK CHARGES": "fees", "CRUK": "fees",
                 "M": "manual posting", "B": "bad debt adjustment", "D": "discount", "S": "samples"}
data["PostingType"] = data["StockCode"].map(POSTING_TYPES).where(~data["IsProduct"], "product").fillna("other")

print(f"{len(data):,} invoice lines, months: {data['Month'].min()} – {data['Month'].max()}")
data.groupby("Month")["Value"].sum().plot.bar(title="Net revenue per month (GBP) – December 2011 is incomplete", figsize=(10, 3));

## 3. The tools

A tool is a normal Python function. ADK reads the **name, the type hints and the docstring** and passes them to the LLM as a tool
description – so the docstring is effectively part of the prompt and should explain clearly what the function does.

Rules for good tools (especially for small local models):

* few, simple parameters (here only `month` in the format `YYYY-MM`),
* return a small dictionary with rounded numbers – not a whole data frame,
* return an `error` message instead of raising an exception, so the agent can react.

In [ ]:
def _r(x):
    return round(float(x), 2)

def _check_month(month: str):
    if month not in set(data["Month"]):
        return {"error": f"Unknown month '{month}'. Use format YYYY-MM between {data['Month'].min()} and {data['Month'].max()}."}
    return None

def _previous_month(month: str) -> str:
    return str(pd.Period(month, "M") - 1)

def _product_sales(month: str) -> pd.DataFrame:
    m = data[data["Month"] == month]
    return m[m["IsProduct"] & ~m["IsCancellation"]]


def get_monthly_kpis(month: str) -> dict:
    '''Returns the key figures of one month: net revenue, product sales, cancellations, other postings
    (shipping, fees, manual postings), number of invoices and customers, average order value and the top 3 countries.
    month: month in the format YYYY-MM, e.g. "2011-11".'''
    if (err := _check_month(month)):
        return err
    m = data[data["Month"] == month]
    sales = _product_sales(month)
    countries = sales.groupby("Country")["Value"].sum().nlargest(3)
    return {
        "month": month,
        "net_revenue_GBP": _r(m["Value"].sum()),
        "product_sales_GBP": _r(sales["Value"].sum()),
        "cancellations_GBP": _r(m.loc[m["IsCancellation"], "Value"].sum()),
        "other_postings_GBP": _r(m.loc[~m["IsProduct"] & ~m["IsCancellation"], "Value"].sum()),
        "number_of_invoices": int(sales["InvoiceNo"].nunique()),
        "number_of_customers": int(sales["CustomerID"].nunique()),
        "average_order_value_GBP": _r(sales["Value"].sum() / sales["InvoiceNo"].nunique()),
        "top_3_countries_GBP": {c: _r(v) for c, v in countries.items()},
    }


def compare_months(month: str, previous_month: str) -> dict:
    '''Variance analysis of product sales of a month compared with an earlier month. Splits the change into a volume
    effect (more or fewer units sold), a price effect (different average prices) and the effect of new and
    discontinued products. Also returns the 3 countries and the 3 products with the largest change.
    month: the month to analyse, format YYYY-MM, e.g. "2011-11".
    previous_month: the month to compare with, usually the month before, e.g. "2011-10".'''
    prev = previous_month
    for mo in (month, prev):
        if (err := _check_month(mo)):
            return err
    cur, old = _product_sales(month), _product_sales(prev)
    agg = lambda d: d.groupby("StockCode").agg(q=("Quantity", "sum"), v=("Value", "sum"))
    p = agg(old).join(agg(cur), how="outer", lsuffix="0", rsuffix="1").fillna(0)
    both = p[(p["q0"] > 0) & (p["q1"] > 0)]
    price0, price1 = both["v0"] / both["q0"], both["v1"] / both["q1"]
    volume_effect = ((both["q1"] - both["q0"]) * price0).sum()
    price_effect = ((price1 - price0) * both["q1"]).sum()
    new_products = p.loc[p["q0"] == 0, "v1"].sum()
    discontinued = -p.loc[p["q1"] == 0, "v0"].sum()

    country = (cur.groupby("Country")["Value"].sum()
               .sub(old.groupby("Country")["Value"].sum(), fill_value=0))
    names = data.drop_duplicates("StockCode").set_index("StockCode")["Description"]
    product = p["v1"] - p["v0"]
    top_products = product.reindex(product.abs().nlargest(3).index)
    return {
        "month": month, "previous_month": prev,
        "product_sales_GBP": _r(cur["Value"].sum()),
        "product_sales_previous_month_GBP": _r(old["Value"].sum()),
        "change_GBP": _r(cur["Value"].sum() - old["Value"].sum()),
        "change_percent": _r(100 * (cur["Value"].sum() / old["Value"].sum() - 1)),
        "volume_effect_GBP": _r(volume_effect),
        "price_effect_GBP": _r(price_effect),
        "new_products_GBP": _r(new_products),
        "discontinued_products_GBP": _r(discontinued),
        "largest_changes_by_country_GBP": {c: _r(v) for c, v in country.reindex(country.abs().nlargest(3).index).items()},
        "largest_changes_by_product_GBP": {f"{k} {names.get(k, '')}".strip(): _r(v) for k, v in top_products.items()},
    }

In [ ]:
def find_cancellations(month: str) -> dict:
    '''Audit check: analyses the cancelled invoices of a month. Returns number and value of cancellations,
    the cancellation rate (cancelled value in % of product sales) compared with the previous month,
    and the 3 customers with the highest cancelled value.
    month: month in the format YYYY-MM, e.g. "2011-11".'''
    if (err := _check_month(month)):
        return err
    def rate(mo):
        m = data[data["Month"] == mo]
        sales = _product_sales(mo)["Value"].sum()
        return 100 * -m.loc[m["IsCancellation"], "Value"].sum() / sales if sales else None
    m = data[(data["Month"] == month) & data["IsCancellation"]]
    prev = _previous_month(month)
    customers = m.groupby(m["CustomerID"].fillna(-1).astype(int))["Value"].sum().nsmallest(3)
    return {
        "month": month,
        "cancelled_invoices": int(m["InvoiceNo"].nunique()),
        "cancelled_value_GBP": _r(m["Value"].sum()),
        "cancellation_rate_percent": _r(rate(month)),
        "cancellation_rate_previous_month_percent": _r(rate(prev)) if prev in set(data["Month"]) else None,
        "top_3_customers_by_cancelled_value_GBP": {("unknown" if c == -1 else str(c)): _r(v) for c, v in customers.items()},
    }


def find_manual_postings(month: str) -> dict:
    '''Audit check: lists all non-product postings of a month (shipping, fees, manual postings, discounts,
    bad debt adjustments, samples) grouped by type with number and value, plus the 5 largest single manual
    postings, discounts and adjustments, which should be checked by the controller.
    month: month in the format YYYY-MM, e.g. "2011-11".'''
    if (err := _check_month(month)):
        return err
    m = data[(data["Month"] == month) & ~data["IsProduct"]]
    by_type = m.groupby("PostingType")["Value"].agg(["size", "sum"])
    critical = m[m["PostingType"].isin(["manual posting", "bad debt adjustment", "discount", "other"])]
    largest = critical.reindex(critical["Value"].abs().nlargest(5).index)
    return {
        "month": month,
        "postings_by_type": {t: {"count": int(r["size"]), "value_GBP": _r(r["sum"])} for t, r in by_type.iterrows()},
        "largest_manual_postings_to_check": [
            {"invoice": r.InvoiceNo, "date": str(r.InvoiceDate.date()), "type": r.PostingType,
             "value_GBP": _r(r.Value), "customer": "unknown" if pd.isna(r.CustomerID) else str(int(r.CustomerID))}
            for r in largest.itertuples()],
    }


def find_unusual_orders(month: str) -> dict:
    '''Audit check: finds unusually large orders of a month (invoice value above the 99.5% quantile of all invoices
    of the year) and measures the share of invoice lines and revenue without a customer number.
    month: month in the format YYYY-MM, e.g. "2011-11".'''
    if (err := _check_month(month)):
        return err
    sales_all = data[data["IsProduct"] & ~data["IsCancellation"]]
    invoices = sales_all.groupby("InvoiceNo").agg(value=("Value", "sum"), month=("Month", "first"),
                                                  customer=("CustomerID", "first"), country=("Country", "first"))
    threshold = invoices["value"].quantile(0.995)
    large = invoices[(invoices["month"] == month) & (invoices["value"] > threshold)].nlargest(5, "value")
    sales = _product_sales(month)
    no_customer = sales["CustomerID"].isna()
    return {
        "month": month,
        "threshold_large_order_GBP": _r(threshold),
        "number_of_large_orders": int(((invoices["month"] == month) & (invoices["value"] > threshold)).sum()),
        "largest_orders": [{"invoice": i, "value_GBP": _r(r.value), "country": r.country,
                            "customer": "unknown" if pd.isna(r.customer) else str(int(r.customer))}
                           for i, r in large.iterrows()],
        "share_of_lines_without_customer_percent": _r(100 * no_customer.mean()),
        "revenue_without_customer_GBP": _r(sales.loc[no_customer, "Value"].sum()),
    }

### Test the tools without AI

Tools are ordinary functions – they can (and should) be tested before an LLM uses them.

In [ ]:
TEST_CALLS = [(get_monthly_kpis, ("2011-11",)), (compare_months, ("2011-11", "2011-10")),
              (find_cancellations, ("2011-11",)), (find_manual_postings, ("2011-11",)), (find_unusual_orders, ("2011-11",))]
for tool, args in TEST_CALLS:
    print(f"--- {tool.__name__}{args}")
    print(json.dumps(tool(*args), indent=1, ensure_ascii=False))

## 4. A single agent: the controlling assistant

An ADK `Agent` consists of

* a **model** – here our local Ollama model,
* an **instruction** – the system prompt that describes role and rules,
* a list of **tools**.

ADK runs the *agent loop*: the LLM decides to call a tool → ADK executes the Python function → the result goes back to the LLM →
the LLM calls further tools or writes the final answer.

In [ ]:
assistant = Agent(
    name="controlling_assistant",
    model=llm,
    description="Answers questions about revenue, cancellations and postings of the online shop.",
    instruction=(
        "You are an assistant for the controlling department of a UK online gift shop. "
        "Answer questions about monthly figures by calling your tools. Months have the format YYYY-MM. "
        "Use ONLY numbers returned by the tools, never estimate or invent figures. "
        "Amounts are in GBP. Answer briefly and precisely."
    ),
    tools=[get_monthly_kpis, compare_months, find_cancellations, find_manual_postings, find_unusual_orders],
)

A small helper runs an agent and shows **every step**: which agent calls which tool with which arguments, and the final answer.
This transparency is important – we can check that the numbers really come from the tools.

In [ ]:
APP, USER = "controlling_app", "controller"

async def run_agent(agent, question, show_tool_results=False):
    session_service = InMemorySessionService()
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    session = await session_service.create_session(app_name=APP, user_id=USER)
    message = types.Content(role="user", parts=[types.Part(text=question)])
    print(f"QUESTION: {question}\n")
    async for event in runner.run_async(user_id=USER, session_id=session.id, new_message=message):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                print(f"[{event.author}] calls tool {part.function_call.name}({dict(part.function_call.args)})")
            elif part.function_response and show_tool_results:
                print(f"[{event.author}] tool result: {json.dumps(part.function_response.response)[:300]} ...")
            elif part.text and event.is_final_response():
                print(f"\n[{event.author}] ANSWER:\n{part.text}\n")
    return await session_service.get_session(app_name=APP, user_id=USER, session_id=session.id)

In [ ]:
_ = await run_agent(assistant, "What was the net revenue in November 2011 and how many customers did we have?")

In [ ]:
_ = await run_agent(assistant, "Why did product sales change from October to November 2011? Is it more a price or a volume effect?")

In [ ]:
_ = await run_agent(assistant, "Were there unusual manual postings in August 2011?", show_tool_results=True)

### Check the answers critically

Compare the tool calls and results above with the answers. Typical errors of (small) LLMs are:

* the tool is called with the **wrong argument**, e.g. `month="2011-10"` when October should be compared with November,
* amounts are **mixed up or summed up wrongly** when the LLM combines several values of a tool result,
* the LLM **calculates** something itself (shares, sums) – and makes arithmetic mistakes.

Better tool descriptions (docstrings), clear instructions and larger models reduce these errors, but do not eliminate them.
This is why the tool calls are printed and why a human remains responsible for the result.

## 5. The agent team: month-end report

For a complete month-end report we split the work among three specialised agents. A `SequentialAgent` runs them one after the other.
The agents pass their results via the shared **session state**: `output_key` stores the answer of an agent in the state, and
`{key}` in the instruction of a later agent inserts it.

| agent | tools | writes to state |
|---|---|---|
| `kpi_analyst` | `get_monthly_kpis`, `compare_months` | `kpi_analysis` |
| `auditor` | `find_cancellations`, `find_manual_postings`, `find_unusual_orders` | `audit_findings` |
| `report_writer` | – (only uses `{kpi_analysis}` and `{audit_findings}`) | `management_report` |

Each agent has only the tools it needs – this keeps the prompts short, which is particularly important for small local models.

In [ ]:
kpi_analyst = Agent(
    name="kpi_analyst",
    model=llm,
    instruction=(
        "You are a KPI analyst in controlling. The user names a month (format YYYY-MM). "
        "Call get_monthly_kpis for this month and compare_months with month = this month and "
        "previous_month = the month before (e.g. month='2011-11', previous_month='2011-10'). "
        "Then summarise in at most 6 bullet points: net revenue, product sales and change vs. previous month, "
        "volume effect, price effect, new products, the most important countries/products driving the change. "
        "Use ONLY numbers from the tools."
    ),
    tools=[get_monthly_kpis, compare_months],
    output_key="kpi_analysis",
)

auditor = Agent(
    name="auditor",
    model=llm,
    instruction=(
        "You are an internal auditor. For the month the user asked for, you MUST call all three tools: "
        "find_cancellations, find_manual_postings and find_unusual_orders. Do not write any text before all three "
        "tools have returned their results. "
        "Then list at most 5 findings that the controller should check, each with the relevant amount in GBP "
        "(e.g. rising cancellation rate, large manual postings or adjustments, very large orders, revenue without customer number). "
        "Use ONLY numbers from the tools."
    ),
    tools=[find_cancellations, find_manual_postings, find_unusual_orders],
    output_key="audit_findings",
)

report_writer = Agent(
    name="report_writer",
    model=llm,
    instruction=(
        "You are the head of controlling and write the management commentary for the month-end report.\n\n"
        "KPI analysis:\n{kpi_analysis}\n\n"
        "Audit findings:\n{audit_findings}\n\n"
        "Write a short report (max. 200 words) with the sections: 1. Key figures, 2. Explanation of the development, "
        "3. Open points to clarify. Use only the figures above, do not add new numbers."
    ),
    output_key="management_report",
)

month_end_team = SequentialAgent(
    name="month_end_closing",
    sub_agents=[kpi_analyst, auditor, report_writer],
)

In [ ]:
session = await run_agent(month_end_team, "Prepare the month-end report for 2011-11.")

### The session state

All intermediate results are stored in the session state – useful for documentation and for checking the work of each agent.

In [ ]:
for key in ["kpi_analysis", "audit_findings", "management_report"]:
    print(f"===== {key} =====\n{session.state.get(key)}\n")

### Fact check: are the numbers in the report correct?

LLMs can make mistakes – small local models in particular. A simple check: all amounts in the report should appear in the tool results.

In [ ]:
import re

tool_numbers = set()
for tool, args in TEST_CALLS:      # the tool calls for 2011-11 from section 3
    tool_numbers |= {abs(float(x)) for x in re.findall(r"-?\d+\.?\d*", json.dumps(tool(*args)))}

def unverified_numbers(text):
    numbers = [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", text or "")]
    # ignore small numbers such as list numbering, days and months, and the years
    return numbers, [x for x in numbers if x > 31 and x not in (2010, 2011)
                     and not any(abs(x - t) <= 0.001 * t + 0.01 for t in tool_numbers)]

for key in ["kpi_analysis", "audit_findings", "management_report"]:
    numbers, unverified = unverified_numbers(session.state.get(key))
    print(f"{key}: {len(numbers)} numbers, not found in the tool results: {unverified or 'none'}")

Numbers that are not found were either **calculated by the LLM itself** (e.g. shares in %) or are **wrong**. Both must be checked
by a human before the report is published. If a figure is needed regularly, it belongs in a tool, not in the LLM.

## Summary

* An agent = LLM + instruction + tools. Google ADK runs the agent loop; with LiteLLM and Ollama the LLM runs **locally** – no data leaves the company.
* The **tools** contain the business logic (KPI definitions, variance analysis, audit rules) and are ordinary, testable Python functions.
  The LLM orchestrates them and writes the text.
* Several **specialised agents** with few tools each are more robust than one agent with many tools – a `SequentialAgent` is the simplest
  form of a multi-agent system.
* Still: the result must be checked. Small models sometimes skip a tool, round numbers or draw wrong conclusions → transparency
  (showing tool calls) and simple automatic checks are essential.

## Exercises

1. Run the month-end report for another month, e.g. `2011-09` or `2011-08` (bad debt adjustments!).
2. Write a new tool `get_top_customers(month)` and give it to the KPI analyst. Adjust the instruction.
3. Add a fourth agent `reviewer` that compares the management report with `{kpi_analysis}` and `{audit_findings}` and lists
   all statements that are not supported by the figures.
4. Replace the `SequentialAgent` by a `ParallelAgent` for the analyst and the auditor (both are independent). What changes?
5. Compare different models (`llama3.1:8b`, `qwen3:8b`, `qwen2.5:3b`): which one calls the tools reliably? How long does the report take?